# Notebook to test environment

### This notebook contains excerpts from select 2025 WaterTAP Academy Notebooks to test against the 2026 WaterTAP Academy environment.



## Test 1: Week 3 - `RO_Flowsheet_ERD_Techno_Economics_Analysis.ipynb`

In [ ]:
# Imports from Pyomo
from pyomo.environ import (
    ConcreteModel,
    Var,
    Param,
    Constraint,
    Objective,
    value,
    assert_optimal_termination,
    TransformationFactory,
    units as pyunits,
)
from pyomo.network import Arc

# Imports from IDAES
from idaes.core import FlowsheetBlock
from idaes.models.unit_models import Feed, Product
from idaes.core.util.model_statistics import degrees_of_freedom
from idaes.core.util.scaling import calculate_scaling_factors, set_scaling_factor
from idaes.core.util.initialization import propagate_state

# Imports from WaterTAP
from watertap.property_models.seawater_prop_pack import SeawaterParameterBlock
from watertap.unit_models.pressure_changer import Pump
from watertap.unit_models.pressure_changer import EnergyRecoveryDevice
from watertap.unit_models.reverse_osmosis_0D import (
    ReverseOsmosis0D,
    ConcentrationPolarizationType,
    MassTransferCoefficient,
    PressureChangeType,
)

# Import costing package
from watertap.costing import WaterTAPCosting
from idaes.core import UnitModelCostingBlock

from watertap.core.solvers import get_solver

# Create a model object using ConcreteModel
m = ConcreteModel()
# Add the flowsheet to the model using FlowsheetBlock
m.fs = FlowsheetBlock(dynamic=False)
# Add the Seawater property package
m.fs.properties = SeawaterParameterBlock()
# Add feed
m.fs.feed = Feed(property_package=m.fs.properties)
# Set feed stream
m.fs.feed.properties[0].flow_vol_phase["Liq"].fix(1e-3)
m.fs.feed.properties[0].conc_mass_phase_comp["Liq", "TDS"].fix(35)
m.fs.feed.properties[0].pressure.fix(101325)
m.fs.feed.properties[0].temperature.fix(273.15 + 25)


# Add pump
m.fs.pump = Pump(property_package=m.fs.properties)
# Set pump parameters
m.fs.pump.efficiency_pump.fix(0.80)
m.fs.pump.control_volume.properties_out[0].pressure.fix(75 * pyunits.bar)


# Add 0D reverse osmosis unit
m.fs.RO = ReverseOsmosis0D(
    property_package=m.fs.properties,
    has_pressure_change=True,
    pressure_change_type=PressureChangeType.calculated,
    mass_transfer_coefficient=MassTransferCoefficient.calculated,
    concentration_polarization_type=ConcentrationPolarizationType.calculated,
)
# Set RO parameters
m.fs.RO.A_comp.fix(4.2e-12)
m.fs.RO.B_comp.fix(3.5e-8)
m.fs.RO.recovery_vol_phase[0, "Liq"].fix(0.5)
m.fs.RO.feed_side.channel_height.fix(1e-3)
m.fs.RO.feed_side.spacer_porosity.fix(0.97)
m.fs.RO.permeate.pressure[0].fix(101325)
m.fs.RO.area.fix(50)

# Add Energy Recovery Device
m.fs.erd = EnergyRecoveryDevice(property_package=m.fs.properties)
# Set ERD parameters
m.fs.erd.efficiency_pump.fix(0.95)
m.fs.erd.outlet.pressure[0].fix(101325)

# Add product stream
m.fs.product = Product(property_package=m.fs.properties)
# Define the connectivity using Arcs
m.fs.feed_to_pump = Arc(source=m.fs.feed.outlet, destination=m.fs.pump.inlet)
m.fs.pump_to_RO = Arc(source=m.fs.pump.outlet, destination=m.fs.RO.inlet)
m.fs.RO_to_erd = Arc(source=m.fs.RO.retentate, destination=m.fs.erd.inlet)
m.fs.RO_to_product = Arc(source=m.fs.RO.permeate, destination=m.fs.product.inlet)

# Use the TransformationFactory to expand the arcs
TransformationFactory("network.expand_arcs").apply_to(m)
# Set Scaling factors
m.fs.properties.set_default_scaling("flow_mass_phase_comp", 1, index=("Liq", "H2O"))
m.fs.properties.set_default_scaling("flow_mass_phase_comp", 1e2, index=("Liq", "TDS"))
set_scaling_factor(m.fs.pump.control_volume.work, 1e-3)
set_scaling_factor(m.fs.RO.area, 1e-2)
calculate_scaling_factors(m)
# Get WaterTAP solver
solver = get_solver()
solver.solve(m.fs.feed)
# Propagate state from feed to pump
propagate_state(m.fs.feed_to_pump)
# Initialize pump
m.fs.pump.initialize()
# Propagate state from pump to RO
propagate_state(m.fs.pump_to_RO)
# Initialize RO
m.fs.RO.initialize()
# Propagate state from RO to ERD and product
propagate_state(m.fs.RO_to_erd)
propagate_state(m.fs.RO_to_product)
# Initialize the product and ERD
m.fs.product.initialize()
m.fs.erd.initialize()
print("DOF = ", degrees_of_freedom(m))
# Solve the model and assert the the model has reached optimal termination
results = solver.solve(m, tee=False)
assert_optimal_termination(results)
print(
    f"Solver Status: {results.solver.status}, Termination Condition: {results.solver.termination_condition}"
)
# Report unit-level performance results for (1) the pump, (2) RO and (3) the energy recovery device, using the report() method for each unit on the flowsheet.
m.fs.pump.report()
m.fs.RO.report()
m.fs.erd.report()
# Add costing
m.fs.costing = WaterTAPCosting()
m.fs.costing.base_currency = pyunits.USD_2020

m.fs.pump.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.costing)
m.fs.RO.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.costing)
m.fs.erd.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.costing)

# Process costing
m.fs.costing.cost_process()
# Add Annual water production
m.fs.costing.add_annual_water_production(m.fs.product.properties[0].flow_vol)

# Levelized Cost of Water (LCOW)
m.fs.costing.add_LCOW(m.fs.product.properties[0].flow_vol)

# Specific Energy Consumption (SEC)
m.fs.costing.add_specific_energy_consumption(m.fs.product.properties[0].flow_vol)
# Initialize costing
m.fs.costing.initialize()

# ReSolve the model and assert the the model has reached optimal termination
results = solver.solve(m, tee=False)
assert_optimal_termination(results)
print(
    f"Solver Status: {results.solver.status}, Termination Condition: {results.solver.termination_condition}"
)

# Display key metrics : LCOW
print("Levelized cost of water: %.2f $/m3" % value(m.fs.costing.LCOW))

# Display key metrics : SEC
print(
    "Specific energy consumption: %.1f kWh/m3"
    % value(m.fs.costing.specific_energy_consumption)
)

# Display other metrics
print("Total operating cost: %.4g $/yr" % value(m.fs.costing.total_operating_cost))
print("Total capital cost: %.4g $" % value(m.fs.costing.total_capital_cost))
print("Total annualized cost: %.4g $/yr" % value(m.fs.costing.total_annualized_cost))

## Test 2: Week 6 - `parmest_ro.ipynb`

In [ ]:
# === Pyomo Imports ===
from pyomo.environ import (
    ConcreteModel,
    value,
    TransformationFactory,
    assert_optimal_termination,
)
from pyomo.network import Arc
import pyomo.contrib.parmest.parmest as parmest

# === IDAES Imports ===
from idaes.core import FlowsheetBlock
from idaes.models.unit_models import Feed, Separator
from idaes.core.util.initialization import propagate_state
from idaes.core.util.model_statistics import degrees_of_freedom
import idaes.core.util.scaling as iscale
import idaes.logger as idaeslog

# === WaterTAP Imports ===
from watertap.unit_models.reverse_osmosis_0D import (
    ReverseOsmosis0D as RO,
    ConcentrationPolarizationType,
    MassTransferCoefficient,
)
from watertap.property_models import seawater_prop_pack as props
from watertap.core.solvers import get_solver

# === Data & Visualization ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


def load_data(full_data):
    # # Load data
    psi_to_pascal = 6894.75  # Pressure conversion
    gpm_to_m3ps = 6.309e-005  # Vol. flow conversion

    dt = full_data[
        [
            "Time",
            "RO Overall Permeate Flowrate (gpm)",
            "RO Overall Concentrate Flowrate (gpm)",
            "Concentrate Recirculation Flowrate Set Point (gpm)",
            "CF Effluent EC (uS/cm)",
            "CF Effluent EC (ppm)",
            "RO Overall Concentrate EC (uS/cm)",
            "RO Overall Concentrate EC (ppm)",
            "RO Overall Permeate EC (uS/cm)",
            "RO Overall Permeate EC (ppm)",
            "RO IN Feedwater Pressure (psi)",
            "RO OUT Concentrate Pressure (psi)",
            "CF Effluent Temperature (C)",
            "Recovery (%)",
        ]
    ]
    dt.rename(
        columns={"Concentrate Recirculation Flowrate Set Point (gpm)": "Qr_gpm"},
        inplace=True,
    )
    # RO pressure drop, psi
    dt["RO_deltaP_psi"] = (
        dt["RO IN Feedwater Pressure (psi)"] - dt["RO OUT Concentrate Pressure (psi)"]
    )
    # Influent feed volumetric flowrate, gpm
    dt["Qp+Qb_gpm"] = (
        dt["RO Overall Permeate Flowrate (gpm)"]
        + dt["RO Overall Concentrate Flowrate (gpm)"]
    )
    # RO feed volumetric flowrate, gpm
    dt["Qr+Qp+Qb_gpm"] = dt["Qp+Qb_gpm"] + dt["Qr_gpm"]

    # RO feed mass frac, ppm
    dt["RO_feed_mass_frac"] = (
        1e-6
        * (
            dt["RO Overall Concentrate EC (ppm)"] * dt["Qr_gpm"]
            + dt["CF Effluent EC (ppm)"] * dt["Qp+Qb_gpm"]
        )
        / dt["Qr+Qp+Qb_gpm"]
    )
    # Column naming from legacy naming by Tim/Mayo:
    dt.rename(
        columns={
            "RO Overall Permeate Flowrate (gpm)": "Qp_gpm",
            "RO Overall Concentrate Flowrate (gpm)": "Qb_gpm",
            "RO Overall Concentrate EC (uS/cm)": "Cb_ec_uS_cm",
            "RO Overall Concentrate EC (ppm)": "Cb_ppm",
            "RO Overall Permeate EC (uS/cm)": "Cp_ec_uS_cm",
            "RO Overall Permeate EC (ppm)": "Cp_ppm",
            "RO IN Feedwater Pressure (psi)": "Pf_psi",
            "RO OUT Concentrate Pressure (psi)": "Pb_psi",
            "Recovery (%)": "system_recovery_%",
        },
        inplace=True,
    )
    dt["RO_recovery_%"] = dt["Qp_gpm"] / dt["Qr+Qp+Qb_gpm"]
    # dt["pressure_in"] = dt["Pf_psi"] * psi_to_pascal
    # dt["flow_vol_permeate"] = dt["Qp_gpm"] * gpm_to_m3ps
    dt["mass_frac_TDS_permeate"] = dt["Cp_ppm"] * 1e-6

    dt.rename(
        columns={
            "Qr+Qp+Qb_gpm": "flow_vol_in",
            "Pf_psi": "pressure_in",
            "RO_feed_mass_frac": "mass_frac_TDS_in",
            "RO_deltaP_psi": "deltaP",
            "Qp_gpm": "flow_vol_permeate",
            "Cp_ppm": "conc_frac_TDS_permeate",
        },
        inplace=True,
    )

    subset_cols = [
        "flow_vol_in",
        "mass_frac_TDS_in",
        "pressure_in",
        "deltaP",
        "flow_vol_permeate",
        # "conc_frac_TDS_permeate",
        "mass_frac_TDS_permeate",
    ]
    dt = dt[subset_cols]

    full_data = dt.iloc[11:61]

    data = dt.iloc[31:41]

    return data, full_data


# === Suppress verbose logging ===
import logging
import warnings

logging.getLogger("pyomo").setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore")

print("✓ All modules imported successfully!")
# Load raw data from CSV
raw_data = pd.read_csv("Plant_data.csv")

# Process data (load_data handles rename, filtering, and other pretreatment)
data, full_data = load_data(raw_data)

print("\n--- Data Preview ---")
display(data)
# Initialize solver
solver = get_solver()

# Unit conversion factors
PSI_TO_PA = 6894.75  # psi → Pascal
GPM_TO_M3PS = 6.309e-5  # GPM → m³/s


def ro_parmest(data):
    """
    Build an RO model configured for parameter estimation.

    Args:
        data: DataFrame with a single row of operating conditions

    Returns:
        Pyomo ConcreteModel with 0 degrees of freedom
    """
    # ===== BUILD FLOWSHEET =====
    m = ConcreteModel()
    m.fs = FlowsheetBlock(dynamic=False)
    m.fs.properties = props.SeawaterParameterBlock()

    # Add units
    m.fs.feed = Feed(property_package=m.fs.properties)
    m.fs.RO = RO(
        property_package=m.fs.properties,
        has_pressure_change=True,
        concentration_polarization_type=ConcentrationPolarizationType.none,
        mass_transfer_coefficient=MassTransferCoefficient.none,
    )

    # Connect units
    m.fs.s00 = Arc(source=m.fs.feed.outlet, destination=m.fs.RO.inlet)
    TransformationFactory("network.expand_arcs").apply_to(m)

    # ===== INITIAL CONDITIONS (for initialization) =====
    m.fs.feed.properties[0].flow_vol_phase.fix(GPM_TO_M3PS * 8)
    m.fs.feed.properties[0].temperature.fix(273.15 + 25)  # 25°C
    m.fs.feed.properties[0].pressure.fix(PSI_TO_PA * 188)
    m.fs.feed.properties[0].mass_frac_phase_comp["Liq", "TDS"].fix(0.001)

    # Membrane configuration
    m.fs.RO.area.fix(28.8)
    m.fs.RO.permeate.pressure[0].fix(101325)  # Atmospheric
    m.fs.RO.deltaP.fix(-PSI_TO_PA * 24.6)

    # ===== PARAMETERS TO ESTIMATE =====
    # Initial guesses for A and B coefficients
    m.fs.RO.A_comp[0, "H2O"].fix(5e-12)  # Water permeability
    m.fs.RO.B_comp[0, "TDS"].fix(4e-8)  # Salt permeability

    # ===== SCALING =====
    m.fs.properties.set_default_scaling(
        "flow_mass_phase_comp", 1e1, index=("Liq", "H2O")
    )
    m.fs.properties.set_default_scaling(
        "flow_mass_phase_comp", 1e6, index=("Liq", "TDS")
    )
    iscale.set_scaling_factor(m.fs.RO.area, 1e-1)
    iscale.calculate_scaling_factors(m)

    # ===== INITIALIZE =====
    solver.solve(m.fs.feed)
    propagate_state(m.fs.s00)
    m.fs.RO.initialize(outlvl=idaeslog.ERROR)

    # ===== SET ACTUAL OPERATING CONDITIONS FROM DATA =====
    m.fs.feed.properties[0].flow_vol_phase.fix(
        GPM_TO_M3PS * float(data.iloc[0]["flow_vol_in"])
    )
    m.fs.feed.properties[0].pressure.fix(PSI_TO_PA * float(data.iloc[0]["pressure_in"]))
    m.fs.feed.properties[0].mass_frac_phase_comp["Liq", "TDS"].fix(
        float(data.iloc[0]["mass_frac_TDS_in"])
    )
    m.fs.RO.deltaP.fix(-PSI_TO_PA * float(data.iloc[0]["deltaP"]))

    # Verify DOF = 0
    assert degrees_of_freedom(m) == 0, f"DOF = {degrees_of_freedom(m)}, expected 0"

    return m


print("✓ Model function defined")

# Parameter names (must match exactly how they appear in the model)
theta_names = [
    "fs.RO.A_comp[0, 'H2O']",  # Water permeability coefficient
    "fs.RO.B_comp[0, 'TDS']",  # Salt permeability coefficient
]

print("Parameters to estimate:")
for name in theta_names:
    print(f"  • {name}")


def SSE(m, data):

    # Normalization factors
    flow_std = np.std(GPM_TO_M3PS * full_data["flow_vol_permeate"])
    tds_std = np.std(full_data["mass_frac_TDS_permeate"])

    # Data values
    flow_measured = GPM_TO_M3PS * float(data.iloc[0]["flow_vol_permeate"])
    tds_measured = float(data.iloc[0]["mass_frac_TDS_permeate"])

    # Model predictions
    flow_model = m.fs.RO.mixed_permeate[0.0].flow_vol_phase["Liq"]
    tds_model = m.fs.RO.mixed_permeate[0.0].mass_frac_phase_comp["Liq", "TDS"]

    # Normalized SSE
    expr = ((flow_measured - flow_model) / flow_std) ** 2 + (
        (tds_measured - tds_model) / tds_std
    ) ** 2

    return expr


print("✓ Objective function defined")

# Create parmest Estimator object
pest = parmest.Estimator(
    ro_parmest,  # Model function
    data,  # Training data
    theta_names,  # Parameters to estimate
    SSE,  # Objective function
    tee=False,  # Suppress solver output
)

print("✓ Estimator created")
print(f"  Training scenarios: {len(data)}")
print(f"  Parameters: {len(theta_names)}")

# Solve the parameter estimation problem
print("Solving parameter estimation...")
obj, theta = pest.theta_est()

print("\n" + "=" * 50)
print("PARAMETER ESTIMATION RESULTS")
print("=" * 50)
print(f"\nOptimal objective value: {obj:.6f}")
print("\nEstimated parameters:")
print(f"  A (water permeability): {theta.iloc[0]:.6e} m/Pa/s")
print(f"  B (salt permeability):  {theta.iloc[1]:.6e} m/s")


def ro_opt(theta):
    """
    Build an RO model with the optimal estimated parameters.

    Args:
        theta: Series containing optimal A and B values

    Returns:
        Initialized Pyomo model ready for simulation
    """
    m = ConcreteModel()
    m.fs = FlowsheetBlock(dynamic=False)
    m.fs.properties = props.SeawaterParameterBlock()
    m.fs.feed = Feed(property_package=m.fs.properties)
    m.fs.RO = RO(
        property_package=m.fs.properties,
        has_pressure_change=True,
        concentration_polarization_type=ConcentrationPolarizationType.none,
        mass_transfer_coefficient=MassTransferCoefficient.none,
    )

    m.fs.s00 = Arc(source=m.fs.feed.outlet, destination=m.fs.RO.inlet)
    TransformationFactory("network.expand_arcs").apply_to(m)

    # Initial conditions
    m.fs.feed.properties[0].flow_vol_phase.fix(GPM_TO_M3PS * 8)
    m.fs.feed.properties[0].temperature.fix(273.15 + 25)
    m.fs.feed.properties[0].pressure.fix(PSI_TO_PA * 188)
    m.fs.feed.properties[0].mass_frac_phase_comp["Liq", "TDS"].fix(0.001)

    m.fs.RO.area.fix(28.8)
    m.fs.RO.permeate.pressure[0].fix(101325)
    m.fs.RO.deltaP.fix(-PSI_TO_PA * 24.6)

    # SET OPTIMAL PARAMETERS
    m.fs.RO.A_comp[0, "H2O"].fix(theta.iloc[0])
    m.fs.RO.B_comp[0, "TDS"].fix(theta.iloc[1])

    # Scaling
    m.fs.properties.set_default_scaling(
        "flow_mass_phase_comp", 1e1, index=("Liq", "H2O")
    )
    m.fs.properties.set_default_scaling(
        "flow_mass_phase_comp", 1e6, index=("Liq", "TDS")
    )
    iscale.set_scaling_factor(m.fs.RO.area, 1e-1)
    iscale.calculate_scaling_factors(m)

    # Initialize
    solver.solve(m.fs.feed)
    propagate_state(m.fs.s00)
    m.fs.RO.initialize(outlvl=idaeslog.ERROR)

    assert degrees_of_freedom(m) == 0
    return m


print("✓ Optimal model function defined")


def save_model_results(model_results, m):
    """
    Run model for each data point and save predictions.
    """
    for i in range(model_results.shape[0]):
        # Update operating conditions
        m.fs.feed.properties[0].flow_vol_phase.fix(
            GPM_TO_M3PS * float(model_results.iloc[i]["flow_vol_in"])
        )
        m.fs.feed.properties[0].pressure.fix(
            PSI_TO_PA * float(model_results.iloc[i]["pressure_in"])
        )
        m.fs.feed.properties[0].mass_frac_phase_comp["Liq", "TDS"].fix(
            float(model_results.iloc[i]["mass_frac_TDS_in"])
        )
        m.fs.RO.deltaP.fix(-PSI_TO_PA * float(model_results.iloc[i]["deltaP"]))

        # Solve
        results = solver.solve(m, tee=False)
        assert_optimal_termination(results)

        # Save predictions
        model_results.iloc[i, model_results.columns.get_loc("flow_vol_permeate")] = (
            value(m.fs.RO.mixed_permeate[0.0].flow_vol_phase["Liq"]) / GPM_TO_M3PS
        )
        model_results.iloc[
            i, model_results.columns.get_loc("mass_frac_TDS_permeate")
        ] = value(m.fs.RO.mixed_permeate[0.0].mass_frac_phase_comp["Liq", "TDS"])

    return model_results


# Initialize model with optimal parameters and generate predictions
m = ro_opt(theta)
model_results = full_data.copy()
model_results = save_model_results(model_results, m)

print("✓ Model predictions saved for all data points")

fig, ax = plt.subplots()
ax.scatter(
    full_data.index,
    full_data["flow_vol_permeate"],
    label="Data",
    color="red",
    s=30,
    alpha=0.7,
)
ax.plot(
    model_results.index,
    model_results["flow_vol_permeate"],
    label="Model",
    color="blue",
    linewidth=2,
)
ax.set_xlabel("Time Node", fontsize=12)
ax.set_ylabel("Permeate Flow Rate (GPM)", fontsize=12)
ax.set_title("Model vs Data", fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.close(fig)

## Test 3: Week 7 - `pump_surrogate_pysmo_final.ipynb`

In [ ]:
# Import necessary libraries
# Import statements
import os
import numpy as np
import pandas as pd

# Import Pyomo libraries
from pyomo.environ import (
    ConcreteModel,
    value,
    Var,
    Constraint,
    Set,
    Objective,
    units as pyunits,
)

# Import IDAES libraries
# Specific imports for surrogate modeling
from idaes.core.surrogate.sampling.data_utils import split_training_validation

# Import PySMO surrogate modeling tools
from idaes.core.surrogate.pysmo_surrogate import PysmoPolyTrainer, PysmoSurrogate

# Import plotting functions for surrogate evaluation
from idaes.core.surrogate.plotting.sm_plotter import (
    surrogate_scatter2D,
    surrogate_parity,
    surrogate_residual,
)

# Import Pump dataset

csv_path = "multispeed-ro-pump_head_power_aligned_above_mcsf.csv"

csv_data = pd.read_csv(csv_path)  # load dataset, 408 data points

# Let's take a look at the first few rows of the dataset
full_data = csv_data
full_data.head()
# Filter relevant columns and order columns such as inputs followed by outputs
input_vars = ["Flowrate_norm_global", "RPM_norm"]
output_vars = ["Power_norm_global", "Head_norm_global"]
data = full_data[input_vars + output_vars]

data.head()
# Filter dataset for single-speed pump (e.g., RPM_norm = 1.0)
filtered_df = data[data["RPM_norm"] == 1.0]
data_single = filtered_df.reset_index(drop=True)
print(f"Single-speed pump dataset size: {data_single.shape}")
data_single.head()
# Separate input and output data
input_data = data_single.loc[:, ["Flowrate_norm_global"]]
output_data = data_single.loc[:, ["Head_norm_global", "Power_norm_global"]]
# input_data = data_single.iloc[:, :1]  # first column is input
# output_data = data_single.iloc[:, ]  # second column and last column are outputs

# Define labels, and split training and validation data
# note that PySMO requires that labels are passed as string lists
input_labels = list(input_data.columns)
output_labels = list(output_data.columns)

n_data = data_single[input_labels[0]].size
# 80/20 train/validation split using IDAES utility function
data_single_training, data_single_validation = split_training_validation(
    data_single, 0.8, seed=n_data
)  # seed=n_data for reproducibility

print(f"Training data size: {data_single_training.shape}")
print(f"Validation data size: {data_single_validation.shape}")

data_single_training.head()
# capture long output (not required to use surrogate API)
from io import StringIO
import sys

stream = StringIO()
oldstdout = sys.stdout
sys.stdout = stream

# Create PySMO trainer object
trainer = PysmoPolyTrainer(
    input_labels=input_labels,
    output_labels=output_labels,
    training_dataframe=data_single_training,
)

# Set PySMO options
trainer.config.maximum_polynomial_order = 4
trainer.config.multinomials = True
trainer.config.training_split = 0.8
trainer.config.number_of_crossvalidations = 10

# Train surrogate (calls PySMO through IDAES Python wrapper)
pysmo_surr_expr = trainer.train_surrogate()  # Surrogate expression object

# revert back to normal output capture
sys.stdout = oldstdout

# display first 50 lines and last 50 lines of output
celloutput = stream.getvalue().split("\n")
for line in celloutput[:50]:
    print(line)
print(".")
print(".")
print(".")
for line in celloutput[-50:]:
    print(line)

# Create callable surrogate object

# Set input bounds for surrogate
xmin = data_single_training[input_labels].min()
xmax = data_single_training[input_labels].max()

input_bounds = {label: (xmin[label], xmax[label]) for label in input_labels}

# Call PysmoSurrogate to create surrogate object
pysmo_surr = PysmoSurrogate(pysmo_surr_expr, input_labels, output_labels, input_bounds)


# To save a model to JSON
model = pysmo_surr.save_to_file("pysmo_poly_surrogate.json", overwrite=True)

# To load a model
pysmo_surr = PysmoSurrogate.load_from_file("pysmo_poly_surrogate.json")

pysmo_surr_expr.display_pysmo_results()

# visualize with IDAES surrogate plotting tools for training data
surrogate_scatter2D(
    pysmo_surr,
    data_single_training,
    filename="pysmo_poly_train_scatter2D.pdf",
    show=True,
)
surrogate_parity(
    pysmo_surr, data_single_training, filename="pysmo_poly_train_parity.pdf", show=True
)
surrogate_residual(
    pysmo_surr,
    data_single_training,
    filename="pysmo_poly_train_residual.pdf",
    show=True,
)

os.remove("pysmo_poly_surrogate.json")
os.remove("pysmo_poly_train_scatter2D.pdf")
os.remove("pysmo_poly_train_parity.pdf")
os.remove("pysmo_poly_train_residual.pdf")
os.remove("solution.pickle")

## Test 4: Week 8 - `Chemistry_Model_Integration_Tutorial.ipynb`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import reaktoro

print(f"Reaktoro version: {reaktoro.__version__}")

# Feed composition (g/L)
feed_composition = {
    "Na": 10556e-3,
    "K": 380e-3,
    "Ca": 400e-3,
    "Mg": 1272e-3,
    "Cl": 18980e-3,
    "SO4": 2649e-3,
    "HCO3": 140e-3,
}

feed_pH = 7.9
temperature_C = 20
pressure_Pa = 101325

# Chemical costs ($/kg)
lime_cost = 0.25
soda_cost = 0.40

print("Feed composition loaded.")
print(f"Feed Ca: {feed_composition['Ca']*1000:.0f} mg/L")
print(f"Feed HCO3: {feed_composition['HCO3']*1000:.0f} mg/L")

# Set up Reaktoro chemical system
db = reaktoro.PhreeqcDatabase("pitzer.dat")

solution = reaktoro.AqueousPhase(reaktoro.speciate("H O Na K Ca Mg Cl S C"))
solution.set(reaktoro.ActivityModelPitzer())

mineral = reaktoro.MineralPhase("Calcite")

system = reaktoro.ChemicalSystem(db, solution, mineral)

print(f"Chemical system created with {system.species().size()} species")


def evaluate_softening(lime_dose_mg_L, soda_dose_mg_L, feed_comp, temp_C):
    """
    Evaluate lime-soda softening using Reaktoro equilibrium.

    Args:
        lime_dose_mg_L: Lime (CaO) dose in mg/L
        soda_dose_mg_L: Soda ash (Na2CO3) dose in mg/L
        feed_comp: Dict of feed concentrations in g/L
        temp_C: Temperature in Celsius

    Returns:
        Dict with results or None if equilibration fails
    """
    # Create chemical state
    state = reaktoro.ChemicalState(system)
    state.temperature(temp_C, "celsius")
    state.pressure(pressure_Pa, "Pa")

    # Add 1 kg of water (≈ 1 L basis)
    state.set("H2O", 1.0, "kg")

    # Add feed components (g/L → mol/L for 1 L basis)
    state.set("Na+", feed_comp["Na"] / 22.99, "mol")
    state.set("K+", feed_comp["K"] / 39.10, "mol")
    state.set("Ca+2", feed_comp["Ca"] / 40.08, "mol")
    state.set("Mg+2", feed_comp["Mg"] / 24.31, "mol")
    state.set("Cl-", feed_comp["Cl"] / 35.45, "mol")
    state.set("SO4-2", feed_comp["SO4"] / 96.06, "mol")
    state.set("HCO3-", feed_comp["HCO3"] / 61.02, "mol")

    # Add lime: CaO + H2O → Ca²⁺ + 2OH⁻
    lime_mol = lime_dose_mg_L * 1e-3 / 56.08  # mg/L → mol/L
    if lime_mol > 0:
        state.add("Ca+2", lime_mol, "mol")
        state.add("OH-", 2 * lime_mol, "mol")

    # Add soda ash: Na₂CO₃ → 2Na⁺ + CO₃²⁻
    soda_mol = soda_dose_mg_L * 1e-3 / 105.99  # mg/L → mol/L
    if soda_mol > 0:
        state.add("Na+", 2 * soda_mol, "mol")
        state.add("CO3-2", soda_mol, "mol")

    # Feed Ca for removal calculation
    feed_ca_mol = feed_comp["Ca"] / 40.08

    # Equilibrate
    solver = reaktoro.EquilibriumSolver(system)

    try:
        result = solver.solve(state)
        if not result.succeeded():
            return None

        # Extract results
        props = reaktoro.ChemicalProps(state)
        aqueous_props = reaktoro.AqueousProps(props)

        effluent_ca_mol = float(state.speciesAmount("Ca+2"))

        # Calcium removal % based on FEED Ca only
        ca_removal = 100 * (feed_ca_mol - effluent_ca_mol) / feed_ca_mol
        ca_removal = max(0, min(100, ca_removal))  # Clamp to 0-100%

        effluent_pH = float(aqueous_props.pH())
        alkalinity = (
            float(aqueous_props.alkalinity()) * 50.04 * 1000
        )  # eq/L → mg/L as CaCO₃
        calcite_mol = float(state.speciesAmount("Calcite"))

        # Cost ($/m³)
        cost = (
            lime_dose_mg_L * 1e-6 * lime_cost + soda_dose_mg_L * 1e-6 * soda_cost
        ) * 1000

        return {
            "ca_removal": ca_removal,
            "effluent_pH": effluent_pH,
            "alkalinity": alkalinity,
            "calcite_mol": calcite_mol,
            "cost": cost,
        }

    except Exception as e:
        return None


# Test with 100 mg/L lime and 200 mg/L soda ash
test = evaluate_softening(100, 200, feed_composition, temperature_C)

if test:
    print("Test evaluation at Lime=100 mg/L, Soda=200 mg/L:")
    print(f"  Ca removal:    {test['ca_removal']:.1f}%")
    print(f"  Effluent pH:   {test['effluent_pH']:.2f}")
    print(f"  Alkalinity:    {test['alkalinity']:.0f} mg/L as CaCO₃")
    print(f"  Cost:          ${test['cost']:.4f}/m³")
else:
    print("Equilibration failed!")

# Define sweep ranges
n_points = 25  # 25×25 = 625 evaluations

lime_range = np.linspace(0, 1200, n_points)  # mg/L
soda_range = np.linspace(0, 1200, n_points)  # mg/L

# Storage arrays
ca_removal_grid = np.zeros((n_points, n_points))
alkalinity_grid = np.zeros((n_points, n_points))
pH_grid = np.zeros((n_points, n_points))
cost_grid = np.zeros((n_points, n_points))

print(f"Running parametric sweep: {n_points}×{n_points} = {n_points**2} evaluations")
print(f"Lime range:  0 - {lime_range[-1]:.0f} mg/L")
print(f"Soda range:  0 - {soda_range[-1]:.0f} mg/L")

# Run the sweep
failed_count = 0

for i, lime_dose in enumerate(lime_range):
    for j, soda_dose in enumerate(soda_range):
        result = evaluate_softening(
            lime_dose, soda_dose, feed_composition, temperature_C
        )

        if result is not None:
            ca_removal_grid[i, j] = result["ca_removal"]
            alkalinity_grid[i, j] = result["alkalinity"]
            pH_grid[i, j] = result["effluent_pH"]
            cost_grid[i, j] = result["cost"]
        else:
            ca_removal_grid[i, j] = np.nan
            alkalinity_grid[i, j] = np.nan
            pH_grid[i, j] = np.nan
            cost_grid[i, j] = np.nan
            failed_count += 1

    if (i + 1) % 5 == 0:
        print(f"  Progress: {100 * (i + 1) / n_points:.0f}%")

print(f"\n✓ Sweep complete. Failed: {failed_count}/{n_points**2}")

## Import Pyomo components
from pyomo.environ import (
    Var,
    Param,
    Constraint,
    Expression,
    Objective,
    ConcreteModel,
    value,
    assert_optimal_termination,
    check_optimal_termination,
    units as pyunits,
)

# IDAES components
from idaes.core import FlowsheetBlock
from idaes.core.util.scaling import set_scaling_factor, constraint_scaling_transform
from idaes.core.util.model_statistics import degrees_of_freedom
from idaes.core.solvers import get_solver

# WaterTAP components
from watertap.property_models.multicomp_aq_sol_prop_pack import MCASParameterBlock
from watertap.unit_models.stoichiometric_reactor import StoichiometricReactor

# Import reaktoro-pse components
from reaktoro_pse.reaktoro_block import ReaktoroBlock
from reaktoro_pse.core.util_classes.cyipopt_solver import get_cyipopt_watertap_solver

m = ConcreteModel()
m.fs = FlowsheetBlock(dynamic=False)
m.fs.properties = MCASParameterBlock(
    solute_list=["Na_+", "K_+", "Ca_2+", "Mg_2+", "Cl_-", "SO4_2-", "HCO3_-"]
)

m.fs.unit = StoichiometricReactor(
    property_package=m.fs.properties,
    precipitate={
        "Calcite": {
            "mw": 100.09 * pyunits.g / pyunits.mol,
            "precipitation_stoichiometric": {"Ca_2+": 1, "HCO3_-": 1},
        },
    },
    # Define reagents
    reagent={
        "Na2CO3": {
            "mw": 105.99 * pyunits.g / pyunits.mol,
            "dissolution_stoichiometric": {"Na_+": 2, "HCO3_-": 1},
        },
        "CaO": {
            "mw": 56.0774 * pyunits.g / pyunits.mol,
            "dissolution_stoichiometric": {"Ca_2+": 1, "H2O": 1},
        },
    },
)
print("StoichiometricReactor model built.")

# Set inlet state
m.fs.unit.dissolution_reactor.properties_in.calculate_state(
    var_args={
        ("flow_vol_phase", "Liq"): 1e-2,  # m³/s = 10 L/s
        ("conc_mass_phase_comp", ("Liq", "Na_+")): 10556 * pyunits.mg / pyunits.L,
        ("conc_mass_phase_comp", ("Liq", "K_+")): 380 * pyunits.mg / pyunits.L,
        ("conc_mass_phase_comp", ("Liq", "Ca_2+")): 400 * pyunits.mg / pyunits.L,
        ("conc_mass_phase_comp", ("Liq", "Mg_2+")): 1272 * pyunits.mg / pyunits.L,
        ("conc_mass_phase_comp", ("Liq", "Cl_-")): 18980 * pyunits.mg / pyunits.L,
        ("conc_mass_phase_comp", ("Liq", "SO4_2-")): 2649 * pyunits.mg / pyunits.L,
        ("conc_mass_phase_comp", ("Liq", "HCO3_-")): 140 * pyunits.mg / pyunits.L,
    },
    hold_state=True,
)

m.fs.unit.dissolution_reactor.properties_in[0].assert_electroneutrality(
    defined_state=True, adjust_by_ion="Cl_-"
)

m.fs.unit.inlet.pressure[0].fix(101325)
m.fs.unit.inlet.temperature[0].fix(273.15 + 20)

print("Feed conditions set.")

# Fix operating conditions (user must guess precipitate amount)
m.fs.unit.reagent_dose["Na2CO3"].fix(1e-3)  # kg/m³
m.fs.unit.reagent_dose["CaO"].fix(1e-3)  # kg/m³
m.fs.unit.flow_mass_precipitate["Calcite"].fix(1e-3)  # kg/s — THIS IS A GUESS!
m.fs.unit.waste_mass_frac_precipitate.fix(0.2)

assert degrees_of_freedom(m.fs.unit) == 0

# Initialize and solve
m.fs.unit.initialize()
solver = get_solver()
results = solver.solve(m.fs.unit, tee=False)
assert_optimal_termination(results)

print("\nBaseline solve with FIXED precipitate (not predicted)")
print(f"This is just mass balance accounting, not real chemistry.")

# Ca removal efficiency
m.fs.unit.ca_removal_efficiency = Expression(
    expr=100
    * (
        m.fs.unit.dissolution_reactor.properties_in[0].flow_mass_phase_comp[
            "Liq", "Ca_2+"
        ]
        - m.fs.unit.precipitation_reactor.properties_out[0].flow_mass_phase_comp[
            "Liq", "Ca_2+"
        ]
    )
    / m.fs.unit.dissolution_reactor.properties_in[0].flow_mass_phase_comp[
        "Liq", "Ca_2+"
    ]
)

# HCO3 removal efficiency
m.fs.unit.hco3_removal_efficiency = Expression(
    expr=100
    * (
        m.fs.unit.dissolution_reactor.properties_in[0].flow_mass_phase_comp[
            "Liq", "HCO3_-"
        ]
        - m.fs.unit.precipitation_reactor.properties_out[0].flow_mass_phase_comp[
            "Liq", "HCO3_-"
        ]
    )
    / m.fs.unit.dissolution_reactor.properties_in[0].flow_mass_phase_comp[
        "Liq", "HCO3_-"
    ]
)

print("Performance metrics added.")

# Variables for Reaktoro interface
m.fs.unit.feed_pH = Var(initialize=7.9)
m.fs.unit.effluent_pH = Var(initialize=7, bounds=(5, 12), units=pyunits.dimensionless)
m.fs.unit.effluent_alkalinity = Var(
    initialize=100, units=pyunits.mg / pyunits.L, bounds=(0.1, None)
)

################## Reaktoro-PSE Interface (Inputs/Outputs) ###################
# Define Reaktoro Outputs
# Outputs from Reaktoro-PSE will be in moles because outputs come directly from Reaktoro
m.fs.unit.reaktoro_outputs = {
    ("speciesAmount", "Calcite"): m.fs.unit.flow_mol_precipitate["Calcite"],
    ("alkalinityAsCaCO3", None): m.fs.unit.effluent_alkalinity,
    ("pH", None): m.fs.unit.effluent_pH,
}

# Build the ReaktoroBlock
# For input, Pyomo vars can be convert to moles as needed
m.fs.unit.eq_precipitation = ReaktoroBlock(
    aqueous_phase={
        "composition": m.fs.unit.dissolution_reactor.properties_in[0].flow_mass_comp,
        "convert_to_rkt_species": True,  # Translate MCAS species' names to Reaktoro names
        "activity_model": reaktoro.ActivityModelPitzer(),
        "fixed_solvent_specie": "H2O",
    },
    system_state={
        "temperature": m.fs.unit.inlet.temperature[0],
        "pressure": m.fs.unit.inlet.pressure[0],
        "pH": m.fs.unit.feed_pH,
    },
    mineral_phase={"phase_components": ["Calcite"]},
    chemistry_modifier={
        "CaO": m.fs.unit.flow_mol_reagent["CaO"],
        "Na2CO3": m.fs.unit.flow_mol_reagent["Na2CO3"],
    },
    outputs=m.fs.unit.reaktoro_outputs,
    database_file="pitzer.dat",
    build_speciation_block=True,
)

print("ReaktoroBlock added — chemistry is now predictive.")

################## Fix Inputs and Model Scaling #############################
# Fix feed pH
m.fs.unit.feed_pH.fix(7.9)

# Switch to molar flow specification for reagents
m.fs.unit.flow_mol_reagent["Na2CO3"].fix(1e-2)  # mol/s
m.fs.unit.flow_mol_reagent["CaO"].fix(1e-2)  # mol/s
m.fs.unit.reagent_dose["Na2CO3"].unfix()
m.fs.unit.reagent_dose["CaO"].unfix()

# KEY STEP: Unfix precipitate — Reaktoro now determines it
m.fs.unit.flow_mass_precipitate["Calcite"].unfix()

# Reaktoro-PSE applies numerical scaling internally in the absence of user-defined scaling factors for inputs and outputs
# Initialize and solve
m.fs.unit.eq_precipitation.initialize()
print("Model Degrees of Freedom = " + str(degrees_of_freedom(m)))

solver = get_cyipopt_watertap_solver(max_iter=200, linear_solver="mumps", pivtol=1e-2)
result = solver.solve(m, tee=True)
assert_optimal_termination(result)

print("\nSolved with chemistry-predicted precipitation.")

print("\n" + "=" * 60)
print("RESULTS: Chemistry-Predicted Precipitation")
print("=" * 60)

print(f"\nReagent Doses:")
print(
    f"  Lime (CaO):        {value(pyunits.convert(m.fs.unit.reagent_dose['CaO'], to_units=pyunits.mg/pyunits.L)):.1f} mg/L"
)
print(
    f"  Soda Ash (Na2CO3): {value(pyunits.convert(m.fs.unit.reagent_dose['Na2CO3'], to_units=pyunits.mg/pyunits.L)):.1f} mg/L"
)

print(f"\nReaktoro-Integrated Predictions:")
print(f"  Effluent pH:       {value(m.fs.unit.effluent_pH):.2f}")
print(f"  Effluent Alk:      {value(m.fs.unit.effluent_alkalinity):.0f} mg/L as CaCO₃")

print(f"\nRemoval Performance:")
print(f"  Ca removal:        {value(m.fs.unit.ca_removal_efficiency):.1f}%")
print(f"  HCO3 removal:      {value(m.fs.unit.hco3_removal_efficiency):.1f}%")
print("=" * 60)

# Reagent cost ($/m³ of feed)
m.fs.reagent_cost = Expression(
    expr=(
        m.fs.unit.flow_mol_reagent["CaO"] * 56.08e-3 * lime_cost
        + m.fs.unit.flow_mol_reagent["Na2CO3"] * 105.99e-3 * soda_cost
    )
    / m.fs.unit.dissolution_reactor.properties_in[0].flow_vol_phase["Liq"]
)

# Unfix reagent flows — these become decision variables
m.fs.unit.flow_mol_reagent["CaO"].unfix()
m.fs.unit.flow_mol_reagent["Na2CO3"].unfix()

# Set bounds
m.fs.unit.flow_mol_reagent["CaO"].setlb(1e-6)
m.fs.unit.flow_mol_reagent["CaO"].setub(10)
m.fs.unit.flow_mol_reagent["Na2CO3"].setlb(1e-5)
m.fs.unit.flow_mol_reagent["Na2CO3"].setub(10)

# Initialize at low doses
m.fs.unit.flow_mol_reagent["CaO"].set_value(1e-5)
m.fs.unit.flow_mol_reagent["Na2CO3"].set_value(1e-5)

# Minimum alkalinity constraint (operational requirement)
m.fs.minimum_alkalinity = Constraint(expr=m.fs.unit.effluent_alkalinity >= 50)

# Target removal constraint (mutable for parametric study)
m.fs.target_removal = Param(initialize=5, mutable=True)
m.fs.target_removal_con = Constraint(
    expr=m.fs.unit.ca_removal_efficiency >= m.fs.target_removal
)

# Objective: minimize cost
m.fs.objective = Objective(expr=m.fs.reagent_cost)
set_scaling_factor(m.fs.objective, 1e2)

print(f"Degrees of Freedom: {degrees_of_freedom(m)}")
print("Optimization problem set up.")

results_list = []

print("\nOptimizing across removal targets...")
print("-" * 70)

for target in range(5, 85, 5):
    m.fs.target_removal.set_value(target)

    if target == 5:
        # Initialize at low doses for first solve
        m.fs.unit.reagent_dose["CaO"].set_value(0.1 * pyunits.mg / pyunits.L)
        m.fs.unit.reagent_dose["Na2CO3"].set_value(0.1 * pyunits.mg / pyunits.L)
        m.fs.unit.eq_precipitation.initialize()

    result = solver.solve(m, tee=False)

    if check_optimal_termination(result):
        results_list.append(
            {
                "target": target,
                "achieved": value(m.fs.unit.ca_removal_efficiency),
                "lime_dose": value(
                    pyunits.convert(
                        m.fs.unit.reagent_dose["CaO"], to_units=pyunits.mg / pyunits.L
                    )
                ),
                "soda_dose": value(
                    pyunits.convert(
                        m.fs.unit.reagent_dose["Na2CO3"],
                        to_units=pyunits.mg / pyunits.L,
                    )
                ),
                "pH": value(m.fs.unit.effluent_pH),
                "cost": value(m.fs.reagent_cost),
                "alkalinity": value(m.fs.unit.effluent_alkalinity),
            }
        )
        print(
            f"Target: {target:3d}% | Achieved: {results_list[-1]['achieved']:5.1f}% | "
            f"Lime: {results_list[-1]['lime_dose']:6.1f} mg/L | "
            f"Soda: {results_list[-1]['soda_dose']:6.1f} mg/L | "
            f"Cost: ${results_list[-1]['cost']:.4f}/m³"
        )
    else:
        print(f"Target: {target:3d}% | INFEASIBLE")

print("-" * 70)
print(f"\nCompleted {len(results_list)} optimizations")